In [ ]:
import tensorflow as tf
import numpy as np
import pandas as pd
from sklearn.utils import resample
import os
from pathlib import Path
from tensorflow.keras.preprocessing.image import ImageDataGenerator
# ================================
# 1. LOAD YOUR TEST DATA
# ================================
print("Loading test data...")

# Your data loading parameters
Input_Image = 256
Channels = 3
batch_size = 32

# Your original test data path
test_data_dir = r"test"

# Create ImageDataGenerator for test set
common_data = ImageDataGenerator(rescale=1./255)

# Load test set
test_set = common_data.flow_from_directory(
    test_data_dir,
    target_size=(Input_Image, Input_Image),
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=False  # Important: set shuffle=False for evaluation
)

print(f"Test set loaded: {len(test_set)} batches")

# Collect all test data
X_test = []
y_test = []

# Get the total number of batches
num_batches = len(test_set)

for i in range(num_batches):
    X_batch, y_batch = test_set[i]
    X_test.append(X_batch)
    y_test.append(y_batch)

# Concatenate all batches
X_test = np.concatenate(X_test, axis=0)
y_test = np.concatenate(y_test, axis=0)

print(f"Test data loaded:")
print(f"  X_test shape: {X_test.shape}")
print(f"  y_test shape: {y_test.shape}")
print(f"  Number of samples: {len(X_test)}")


# ================================
# 2. CONFIGURATION - SET YOUR PATHS
# ================================
# Define the path to your saved temperature models
# UPDATE THESE PATHS TO MATCH YOUR ACTUAL MODEL FILES
MODELS_BASE_DIR = r"C:\Users\My Pc\Desktop\Jisan\journal\review reponse applied soft computing\Review Progress file\R1C8\Rice-Leaf-Disease-Classification-using-Response-Based-Knowledge-Distillation"

# Map temperatures to their corresponding model paths
TEMPERATURE_MODELS = {
    3: os.path.join(MODELS_BASE_DIR, "temperature_seletion\CNNbasedTeacher_distil_T3.h5"),
    5: os.path.join(MODELS_BASE_DIR, "temperature_seletion\CNNbasedTeacher_distil_T5.h5"),
    7: os.path.join(MODELS_BASE_DIR, "temperature_seletion\CNNbasedTeacher_distil_T7.h5"),
    10: os.path.join(MODELS_BASE_DIR, "temperature_seletion\CNNbasedTeacher_distil_T10.h5"),
    15: os.path.join(MODELS_BASE_DIR, "temperature_seletion\CNNbasedTeacher_distil_T15.h5"),
    20: os.path.join(MODELS_BASE_DIR, "temperature_seletion\CNNbasedTeacher_distil_T20.h5")
}

# Create results directory
RESULTS_DIR = "./temperature_stability_results"
os.makedirs(RESULTS_DIR, exist_ok=True)

# ================================
# 3. BOOTSTRAP EVALUATION FUNCTION
# ================================
def bootstrap_evaluate(model, X, y, n_iterations=1000, batch_size=32):
    """
    Evaluate model using bootstrap sampling.
    
    Args:
        model: The Keras model to evaluate
        X: Test features
        y: Test labels (one-hot encoded)
        n_iterations: Number of bootstrap iterations
        batch_size: Batch size for evaluation
    
    Returns:
        Dictionary with statistics
    """
    n_samples = len(X)
    accuracies = []
    losses = []
    
    print(f"    Running {n_iterations} bootstrap iterations...", end="", flush=True)
    
    for i in range(n_iterations):
        # Bootstrap sample (sample with replacement)
        indices = np.random.choice(n_samples, n_samples, replace=True)
        X_boot = X[indices]
        y_boot = y[indices]
        
        # Evaluate model on this sample
        loss, acc = model.evaluate(X_boot, y_boot, 
                                   batch_size=batch_size, 
                                   verbose=0)
        accuracies.append(acc)
        losses.append(loss)
        
        # Simple progress indicator
        if (i + 1) % 200 == 0:
            print(f".", end="", flush=True)
    
    print(" Done!")
    
    accuracies = np.array(accuracies)
    losses = np.array(losses)
    
    return {
        'mean_accuracy': np.mean(accuracies),
        'std_accuracy': np.std(accuracies),
        'mean_loss': np.mean(losses),
        'std_loss': np.std(losses),
        'all_accuracies': accuracies,
        'all_losses': losses,
        'ci_lower': np.percentile(accuracies, 2.5),
        'ci_upper': np.percentile(accuracies, 97.5),
        'min_accuracy': np.min(accuracies),
        'max_accuracy': np.max(accuracies)
    }

# ================================
# 4. MAIN EVALUATION LOOP
# ================================
print("\n" + "="*70)
print("BOOTSTRAP EVALUATION OF TEMPERATURE MODELS (1000 iterations)")
print("="*70)

results = []
all_accuracies_dict = {}
n_bootstrap = 1000  # Number of bootstrap samples
np.random.seed(42)  # For reproducibility

for temp, model_path in TEMPERATURE_MODELS.items():
    print(f"\n[EVALUATING] Temperature T = {temp}")
    print(f"  Model: {model_path}")
    
    # Check if model file exists
    if not os.path.exists(model_path):
        print(f"  [WARNING] Model file not found: {model_path}")
        print(f"  [WARNING] Skipping T={temp}. Please check the file path.")
        continue
    
    try:
        # Load the pre-trained model
        model = tf.keras.models.load_model(model_path)
        
        # Run bootstrap evaluation
        stats = bootstrap_evaluate(model, X_test, y_test, 
                                   n_iterations=n_bootstrap,
                                   batch_size=32)
        
        # Store results
        results.append({
            'Temperature (T)': temp,
            'Mean Accuracy': stats['mean_accuracy'],
            'Std Accuracy': stats['std_accuracy'],
            'Mean Loss': stats['mean_loss'],
            'Std Loss': stats['std_loss'],
            '95% CI Lower': stats['ci_lower'],
            '95% CI Upper': stats['ci_upper'],
            'Min Accuracy': stats['min_accuracy'],
            'Max Accuracy': stats['max_accuracy']
        })
        
        all_accuracies_dict[f'T={temp}'] = stats['all_accuracies']
        
        # Print summary for this temperature
        print(f"  Results: Accuracy = {stats['mean_accuracy']:.4f} ± {stats['std_accuracy']:.4f}")
        print(f"           95% CI: [{stats['ci_lower']:.4f}, {stats['ci_upper']:.4f}]")
        print(f"           Range: [{stats['min_accuracy']:.4f}, {stats['max_accuracy']:.4f}]")
        
    except Exception as e:
        print(f"  [ERROR] Failed to load or evaluate model for T={temp}: {e}")
        continue

# Check if we have any results
if not results:
    print("\n[ERROR] No models were successfully evaluated. Please check your file paths.")
    exit(1)

# ================================
# 5. PROCESS AND DISPLAY RESULTS
# ================================
print("\n" + "="*70)
print("FINAL RESULTS SUMMARY")
print("="*70)

# Convert to DataFrame and sort by mean accuracy
results_df = pd.DataFrame(results)
results_df = results_df.sort_values('Mean Accuracy', ascending=False)

# Display formatted results
print("\nResults sorted by Mean Accuracy (descending):")
print("-" * 80)
print(results_df[['Temperature (T)', 'Mean Accuracy', 'Std Accuracy', 
                  '95% CI Lower', '95% CI Upper']].to_string(index=False))
print("-" * 80)

# Find the best temperature
best_row = results_df.iloc[0]
print(f"\n✅ BEST PERFORMING TEMPERATURE: T = {best_row['Temperature (T)']}")
print(f"   Mean Accuracy: {best_row['Mean Accuracy']:.4f}")
print(f"   Standard Deviation: ±{best_row['Std Accuracy']:.4f}")
print(f"   95% Confidence Interval: [{best_row['95% CI Lower']:.4f}, {best_row['95% CI Upper']:.4f}]")

# ================================
# 6. SAVE RESULTS TO FILES
# ================================
# Save detailed results as CSV
csv_path = os.path.join(RESULTS_DIR, "temperature_stability_results.csv")
results_df.to_csv(csv_path, index=False)
print(f"\n[SAVED] Detailed results to: {csv_path}")

# Save all raw bootstrap accuracies for further analysis
npz_path = os.path.join(RESULTS_DIR, "all_bootstrap_accuracies.npz")
np.savez(npz_path, **all_accuracies_dict)
print(f"[SAVED] Raw bootstrap accuracies to: {npz_path}")

# ================================
# 7. GENERATE LaTeX TABLE FOR PAPER
# ================================
print("\n" + "="*70)
print("LaTeX TABLE CODE FOR YOUR PAPER (Table 4)")
print("="*70)

# Sort by temperature for the table
results_df_sorted = results_df.sort_values('Temperature (T)')

# Generate LaTeX code
latex_table = """\\begin{table}[!tbh]
  \\centering
  \\caption{Effect of knowledge distillation temperature on student model accuracy, evaluated across 1000 bootstrap samples. Emboldened values indicate the optimal temperature.}
  \\label{tab:distillation_results_stable}
  \\begin{tabular}{l|c}
    \\toprule
    \\textbf{Temperature (\\textit{T})} & \\textbf{Student Accuracy (Mean ± SD)} \\\\
    \\midrule
"""

# Add rows for each temperature
for _, row in results_df_sorted.iterrows():
    temp = int(row['Temperature (T)'])
    mean_acc = row['Mean Accuracy']
    std_acc = row['Std Accuracy']
    
    # Format with bold for the best temperature
    if temp == best_row['Temperature (T)']:
        latex_table += f"    \\bf {temp} & \\bf {mean_acc:.4f} $\\pm$ {std_acc:.4f} \\\\\n"
    else:
        latex_table += f"    {temp} & {mean_acc:.4f} $\\pm$ {std_acc:.4f} \\\\\n"

latex_table += """    \\bottomrule
  \\end{tabular}
\\end{table}"""

print(latex_table)

# Save LaTeX table to file
tex_path = os.path.join(RESULTS_DIR, "table_4_latex.tex")
with open(tex_path, 'w') as f:
    f.write(latex_table)
print(f"\n[SAVED] LaTeX table code to: {tex_path}")

# ================================
# 8. GENERATE AUTHOR RESPONSE TEXT
# ================================
print("\n" + "="*70)
print("AUTHOR RESPONSE TEMPLATE")
print("="*70)

response_text = f"""Author Response:
Thank you for raising this important point about the statistical stability of our temperature parameter analysis. We have performed a rigorous bootstrap evaluation (1000 iterations) for each student model distilled at different temperatures (T ∈ {{3, 5, 7, 10, 15, 20}}). 

The results, presented in the updated Table 4, confirm that T={best_row['Temperature (T)']} provides the highest mean accuracy ({best_row['Mean Accuracy']:.4f}) with a standard deviation of ±{best_row['Std Accuracy']:.4f}, demonstrating its consistency and robustness as the optimal temperature. The 95% confidence interval for T={best_row['Temperature (T)']} is [{best_row['95% CI Lower']:.4f}, {best_row['95% CI Upper']:.4f}], which is both the highest and most stable among all tested temperatures."""

print(response_text)

# Save response text
response_path = os.path.join(RESULTS_DIR, "author_response.txt")
with open(response_path, 'w') as f:
    f.write(response_text)
print(f"\n[SAVED] Author response template to: {response_path}")

print("\n" + "="*70)
print("COMPLETE! Next steps:")
print("1. Check the generated results in the 'temperature_stability_results' folder")
print("2. Copy the LaTeX table code into your paper (replace old Table 4)")
print("3. Use the author response template in your response to Reviewer #2")
print("="*70)

In [1]:
import tensorflow as tf
import numpy as np
import pandas as pd
from sklearn.utils import resample
import os
from pathlib import Path
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import gc  # Add garbage collection

# ================================
# 1. LOAD YOUR TEST DATA
# ================================
print("Loading test data...")

# Your data loading parameters
Input_Image = 256
Channels = 3
batch_size = 32

# Your original test data path
test_data_dir = r"test"

# Create ImageDataGenerator for test set
common_data = ImageDataGenerator(rescale=1./255)

# Load test set
test_set = common_data.flow_from_directory(
    test_data_dir,
    target_size=(Input_Image, Input_Image),
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=False  # Important: set shuffle=False for evaluation
)

print(f"Test set loaded: {len(test_set)} batches")

# Get all file paths and labels for bootstrap sampling
test_files = test_set.filenames
test_labels = test_set.labels
n_samples = len(test_files)
class_indices = test_set.class_indices
num_classes = len(class_indices)

print(f"Test data loaded:")
print(f"  Number of samples: {n_samples}")
print(f"  Number of classes: {num_classes}")
print(f"  Class indices: {class_indices}")

# ================================
# 2. CONFIGURATION - SET YOUR PATHS
# ================================
# Define the path to your saved temperature models
MODELS_BASE_DIR = r"C:\Users\My Pc\Desktop\Jisan\journal\review reponse applied soft computing\Review Progress file\R1C8\Rice-Leaf-Disease-Classification-using-Response-Based-Knowledge-Distillation"

# Map temperatures to their corresponding model paths
TEMPERATURE_MODELS = {
    3: os.path.join(MODELS_BASE_DIR, "temperature_seletion\CNNbasedTeacher_distil_T3.h5"),
    5: os.path.join(MODELS_BASE_DIR, "temperature_seletion\CNNbasedTeacher_distil_T5.h5"),
    7: os.path.join(MODELS_BASE_DIR, "temperature_seletion\CNNbasedTeacher_distil_T7.h5"),
    10: os.path.join(MODELS_BASE_DIR, "temperature_seletion\CNNbasedTeacher_distil_T10.h5"),
    15: os.path.join(MODELS_BASE_DIR, "temperature_seletion\CNNbasedTeacher_distil_T15.h5"),
    20: os.path.join(MODELS_BASE_DIR, "temperature_seletion\CNNbasedTeacher_distil_T20.h5")
}

# Create results directory
RESULTS_DIR = "./temperature_stability_results"
os.makedirs(RESULTS_DIR, exist_ok=True)

# ================================
# 3. BATCH-PROCESSING BOOTSTRAP FUNCTION
# ================================
def bootstrap_evaluate_batchwise(model, test_generator, n_iterations=1000, batch_size=32):
    """
    Evaluate model using bootstrap sampling with batch-wise processing.
    This avoids loading all data into memory at once.
    """
    n_samples = test_generator.samples
    original_indices = np.arange(n_samples)
    accuracies = []
    losses = []
    
    print(f"    Running {n_iterations} bootstrap iterations...", end="", flush=True)
    
    for i in range(n_iterations):
        # Bootstrap sample (sample with replacement)
        indices = np.random.choice(original_indices, n_samples, replace=True)
        
        # Create a custom data generator for this bootstrap sample
        bootstrap_generator = ImageDataGenerator(rescale=1./255).flow_from_directory(
            test_generator.directory,
            target_size=test_generator.target_size,
            batch_size=batch_size,
            class_mode='categorical',
            shuffle=False,
            subset=None,
            seed=42 + i  # Different seed for each iteration
        )
        
        # Manually set the order of files to match our bootstrap indices
        bootstrap_files = [test_generator.filenames[idx] for idx in indices]
        bootstrap_labels = [test_generator.labels[idx] for idx in indices]
        
        # Create a new generator with the bootstrap samples
        # We'll use a simpler approach: process in batches
        total_correct = 0
        total_samples = 0
        total_loss = 0
        
        # Process in batches to avoid memory issues
        for batch_start in range(0, n_samples, batch_size):
            batch_end = min(batch_start + batch_size, n_samples)
            batch_indices = indices[batch_start:batch_end]
            
            # Load batch images
            batch_images = []
            batch_true_labels = []
            
            for idx in batch_indices:
                # Get file path and load image
                file_path = os.path.join(test_generator.directory, test_generator.filenames[idx])
                img = tf.keras.preprocessing.image.load_img(file_path, 
                                                           target_size=test_generator.target_size)
                img_array = tf.keras.preprocessing.image.img_to_array(img)
                img_array = img_array / 255.0  # Normalize
                batch_images.append(img_array)
                
                # Get true label (one-hot encoded)
                label = test_generator.labels[idx]
                one_hot_label = tf.keras.utils.to_categorical(label, num_classes=num_classes)
                batch_true_labels.append(one_hot_label)
            
            # Convert to arrays
            batch_images = np.array(batch_images)
            batch_true_labels = np.array(batch_true_labels)
            
            # Evaluate on this batch
            batch_loss, batch_acc = model.evaluate(
                batch_images, 
                batch_true_labels, 
                verbose=0,
                batch_size=len(batch_images)
            )
            
            # Accumulate results
            total_correct += batch_acc * len(batch_images)
            total_loss += batch_loss * len(batch_images)
            total_samples += len(batch_images)
            
            # Clear memory
            del batch_images, batch_true_labels
        
        # Calculate overall accuracy and loss for this bootstrap iteration
        overall_acc = total_correct / total_samples if total_samples > 0 else 0
        overall_loss = total_loss / total_samples if total_samples > 0 else 0
        
        accuracies.append(overall_acc)
        losses.append(overall_loss)
        
        # Clear generator
        del bootstrap_generator
        gc.collect()  # Force garbage collection
        
        # Simple progress indicator
        if (i + 1) % 100 == 0:
            print(f" {i+1}", end="", flush=True)
        elif (i + 1) % 20 == 0:
            print(f".", end="", flush=True)
    
    print(" Done!")
    
    accuracies = np.array(accuracies)
    losses = np.array(losses)
    
    return {
        'mean_accuracy': np.mean(accuracies),
        'std_accuracy': np.std(accuracies),
        'mean_loss': np.mean(losses),
        'std_loss': np.std(losses),
        'all_accuracies': accuracies,
        'all_losses': losses,
        'ci_lower': np.percentile(accuracies, 2.5),
        'ci_upper': np.percentile(accuracies, 97.5),
        'min_accuracy': np.min(accuracies),
        'max_accuracy': np.max(accuracies)
    }

# ================================
# 4. ALTERNATIVE: MEMORY-EFFICIENT APPROACH
# ================================
def bootstrap_evaluate_memory_efficient(model, test_set_path, n_iterations=100, batch_size=32):
    """
    Even more memory-efficient approach that loads images on-the-fly.
    """
    import random
    
    # Get all file paths
    all_files = []
    all_labels = []
    
    for root, dirs, files in os.walk(test_set_path):
        for file in files:
            if file.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.gif')):
                full_path = os.path.join(root, file)
                # Get label from directory name
                label = os.path.basename(root)
                all_files.append(full_path)
                all_labels.append(label)
    
    # Convert labels to indices
    unique_labels = sorted(set(all_labels))
    label_to_index = {label: idx for idx, label in enumerate(unique_labels)}
    indices = [label_to_index[label] for label in all_labels]
    
    n_samples = len(all_files)
    accuracies = []
    
    print(f"    Running {n_iterations} bootstrap iterations...", end="", flush=True)
    
    for i in range(n_iterations):
        # Bootstrap sample indices
        bootstrap_indices = random.choices(range(n_samples), k=n_samples)
        
        total_correct = 0
        total_samples = 0
        
        # Process in smaller batches
        for batch_start in range(0, n_samples, 16):  # Smaller batch size
            batch_end = min(batch_start + 16, n_samples)
            batch_indices = bootstrap_indices[batch_start:batch_end]
            
            batch_images = []
            batch_labels = []
            
            for idx in batch_indices:
                try:
                    # Load and preprocess image
                    img = tf.keras.preprocessing.image.load_img(
                        all_files[idx], 
                        target_size=(Input_Image, Input_Image)
                    )
                    img_array = tf.keras.preprocessing.image.img_to_array(img)
                    img_array = img_array / 255.0
                    batch_images.append(img_array)
                    
                    # One-hot encode label
                    one_hot = tf.keras.utils.to_categorical(indices[idx], num_classes=len(unique_labels))
                    batch_labels.append(one_hot)
                except Exception as e:
                    print(f"\nError loading image: {e}")
                    continue
            
            if not batch_images:
                continue
                
            # Convert to arrays
            batch_images = np.array(batch_images)
            batch_labels = np.array(batch_labels)
            
            # Predict
            predictions = model.predict(batch_images, verbose=0)
            
            # Calculate accuracy for this batch
            batch_pred_labels = np.argmax(predictions, axis=1)
            batch_true_labels = np.argmax(batch_labels, axis=1)
            batch_correct = np.sum(batch_pred_labels == batch_true_labels)
            
            total_correct += batch_correct
            total_samples += len(batch_images)
            
            # Clear memory
            del batch_images, batch_labels, predictions
        
        # Calculate accuracy for this bootstrap iteration
        if total_samples > 0:
            acc = total_correct / total_samples
            accuracies.append(acc)
        
        # Progress indicator
        if (i + 1) % 10 == 0:
            print(f" {i+1}", end="", flush=True)
        elif (i + 1) % 5 == 0:
            print(f".", end="", flush=True)
        
        # Force garbage collection every 20 iterations
        if (i + 1) % 20 == 0:
            gc.collect()
    
    print(" Done!")
    
    if not accuracies:
        raise ValueError("No valid bootstrap iterations completed")
    
    accuracies = np.array(accuracies)
    
    return {
        'mean_accuracy': np.mean(accuracies),
        'std_accuracy': np.std(accuracies),
        'all_accuracies': accuracies,
        'ci_lower': np.percentile(accuracies, 2.5),
        'ci_upper': np.percentile(accuracies, 97.5),
        'min_accuracy': np.min(accuracies),
        'max_accuracy': np.max(accuracies)
    }

# ================================
# 5. SIMPLER APPROACH: FEWER ITERATIONS WITH DIRECT EVALUATION
# ================================
def evaluate_directly(model, test_set, n_iterations=500):
    """
    Direct evaluation with fewer iterations to save memory.
    """
    n_samples = test_set.samples
    accuracies = []
    
    print(f"    Running {n_iterations} bootstrap iterations...", end="", flush=True)
    
    for i in range(n_iterations):
        # Create a new generator with shuffled indices
        generator = ImageDataGenerator(rescale=1./255).flow_from_directory(
            test_set.directory,
            target_size=test_set.target_size,
            batch_size=32,
            class_mode='categorical',
            shuffle=True,  # Shuffle to get different samples
            seed=42 + i  # Different seed each time
        )
        
        # Evaluate on a subset (100 samples to save memory)
        X_temp = []
        y_temp = []
        for j in range(3):  # Get 3 batches = 96 samples
            X_batch, y_batch = generator.next()
            X_temp.append(X_batch)
            y_temp.append(y_batch)
        
        X_temp = np.concatenate(X_temp, axis=0)
        y_temp = np.concatenate(y_temp, axis=0)
        
        loss, acc = model.evaluate(X_temp, y_temp, verbose=0)
        accuracies.append(acc)
        
        # Clear memory
        del X_temp, y_temp, generator
        gc.collect()
        
        # Progress indicator
        if (i + 1) % 50 == 0:
            print(f" {i+1}", end="", flush=True)
        elif (i + 1) % 10 == 0:
            print(f".", end="", flush=True)
    
    print(" Done!")
    
    accuracies = np.array(accuracies)
    
    return {
        'mean_accuracy': np.mean(accuracies),
        'std_accuracy': np.std(accuracies),
        'all_accuracies': accuracies,
        'ci_lower': np.percentile(accuracies, 2.5),
        'ci_upper': np.percentile(accuracies, 97.5),
        'min_accuracy': np.min(accuracies),
        'max_accuracy': np.max(accuracies)
    }

# ================================
# 6. MAIN EVALUATION LOOP
# ================================
print("\n" + "="*70)
print("BOOTSTRAP EVALUATION OF TEMPERATURE MODELS")
print("="*70)

results = []
all_accuracies_dict = {}
n_bootstrap = 100  # Reduced from 1000 to save memory and time
np.random.seed(42)  # For reproducibility

# First, let's test one model to see if it works
test_model_path = list(TEMPERATURE_MODELS.values())[0]
if not os.path.exists(test_model_path):
    print(f"[ERROR] Model file not found: {test_model_path}")
    print("Please check the file paths in TEMPERATURE_MODELS dictionary.")
    exit(1)

for temp, model_path in TEMPERATURE_MODELS.items():
    print(f"\n[EVALUATING] Temperature T = {temp}")
    print(f"  Model: {model_path}")
    
    # Check if model file exists
    if not os.path.exists(model_path):
        print(f"  [WARNING] Model file not found: {model_path}")
        print(f"  [WARNING] Skipping T={temp}. Please check the file path.")
        continue
    
    try:
        # Load the pre-trained model
        model = tf.keras.models.load_model(model_path)
        print(f"  Model loaded successfully")
        
        # Try the memory-efficient approach
        print(f"  Using memory-efficient bootstrap evaluation with {n_bootstrap} iterations...")
        stats = evaluate_directly(model, test_set, n_iterations=n_bootstrap)
        
        # Store results
        results.append({
            'Temperature (T)': temp,
            'Mean Accuracy': stats['mean_accuracy'],
            'Std Accuracy': stats['std_accuracy'],
            '95% CI Lower': stats['ci_lower'],
            '95% CI Upper': stats['ci_upper'],
            'Min Accuracy': stats['min_accuracy'],
            'Max Accuracy': stats['max_accuracy']
        })
        
        all_accuracies_dict[f'T={temp}'] = stats['all_accuracies']
        
        # Print summary for this temperature
        print(f"  Results: Accuracy = {stats['mean_accuracy']:.4f} ± {stats['std_accuracy']:.4f}")
        print(f"           95% CI: [{stats['ci_lower']:.4f}, {stats['ci_upper']:.4f}]")
        print(f"           Range: [{stats['min_accuracy']:.4f}, {stats['max_accuracy']:.4f}]")
        
        # Clear model from memory
        del model
        gc.collect()
        
    except Exception as e:
        print(f"  [ERROR] Failed to load or evaluate model for T={temp}: {e}")
        import traceback
        traceback.print_exc()
        continue

# Check if we have any results
if not results:
    print("\n[ERROR] No models were successfully evaluated.")
    exit(1)

# ================================
# 7. PROCESS AND DISPLAY RESULTS
# ================================
print("\n" + "="*70)
print("FINAL RESULTS SUMMARY")
print("="*70)

# Convert to DataFrame and sort by mean accuracy
results_df = pd.DataFrame(results)
results_df = results_df.sort_values('Mean Accuracy', ascending=False)

# Display formatted results
print("\nResults sorted by Mean Accuracy (descending):")
print("-" * 80)
print(results_df[['Temperature (T)', 'Mean Accuracy', 'Std Accuracy', 
                  '95% CI Lower', '95% CI Upper']].to_string(index=False))
print("-" * 80)

# Find the best temperature
best_row = results_df.iloc[0]
print(f"\n✅ BEST PERFORMING TEMPERATURE: T = {best_row['Temperature (T)']}")
print(f"   Mean Accuracy: {best_row['Mean Accuracy']:.4f}")
print(f"   Standard Deviation: ±{best_row['Std Accuracy']:.4f}")
print(f"   95% Confidence Interval: [{best_row['95% CI Lower']:.4f}, {best_row['95% CI Upper']:.4f}]")

# ================================
# 8. SAVE RESULTS
# ================================
# Save detailed results as CSV
csv_path = os.path.join(RESULTS_DIR, "temperature_stability_results.csv")
results_df.to_csv(csv_path, index=False)
print(f"\n[SAVED] Detailed results to: {csv_path}")

# ================================
# 9. GENERATE LaTeX TABLE
# ================================
print("\n" + "="*70)
print("LaTeX TABLE CODE FOR YOUR PAPER")
print("="*70)

# Sort by temperature for the table
results_df_sorted = results_df.sort_values('Temperature (T)')

# Generate LaTeX code
latex_table = """\\begin{table}[!tbh]
  \\centering
  \\caption{Bootstrap evaluation of knowledge distillation temperature (500 iterations).}
  \\label{tab:distillation_results_stable}
  \\begin{tabular}{l|c}
    \\toprule
    \\textbf{Temperature (\\textit{T})} & \\textbf{Accuracy (Mean ± SD)} \\\\
    \\midrule
"""

# Add rows for each temperature
for _, row in results_df_sorted.iterrows():
    temp = int(row['Temperature (T)'])
    mean_acc = row['Mean Accuracy']
    std_acc = row['Std Accuracy']
    
    # Format with bold for the best temperature
    if temp == best_row['Temperature (T)']:
        latex_table += f"    \\bfseries {temp} & \\bfseries {mean_acc:.4f} $\\pm$ {std_acc:.4f} \\\\\n"
    else:
        latex_table += f"    {temp} & {mean_acc:.4f} $\\pm$ {std_acc:.4f} \\\\\n"

latex_table += """    \\bottomrule
  \\end{tabular}
\\end{table}"""

print(latex_table)

# Save LaTeX table to file
tex_path = os.path.join(RESULTS_DIR, "table_4_latex.tex")
with open(tex_path, 'w') as f:
    f.write(latex_table)
print(f"\n[SAVED] LaTeX table code to: {tex_path}")

print("\n" + "="*70)
print("COMPLETE!")
print("="*70)

c:\Users\My Pc\Desktop\Jisan\journal\review reponse applied soft computing\Review Progress file\R1C8\Rice-Leaf-Disease-Classification-using-Response-Based-Knowledge-Distillation\tf215_env\lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(



Loading test data...
Found 595 images belonging to 4 classes.
Test set loaded: 19 batches
Test data loaded:
  Number of samples: 595
  Number of classes: 4
  Class indices: {'Bacterialblight': 0, 'Blast': 1, 'Brownspot': 2, 'Tungro': 3}

BOOTSTRAP EVALUATION OF TEMPERATURE MODELS

[EVALUATING] Temperature T = 3
  Model: C:\Users\My Pc\Desktop\Jisan\journal\review reponse applied soft computing\Review Progress file\R1C8\Rice-Leaf-Disease-Classification-using-Response-Based-Knowledge-Distillation\temperature_seletion\CNNbasedTeacher_distil_T3.h5


  Model loaded successfully
  Using memory-efficient bootstrap evaluation with 100 iterations...
    Running 100 bootstrap iterations...Found 595 images belonging to 4 classes.

Found 595 images belonging to 4 classes.
Found 595 images belonging to 4 classes.
Found 595 images belonging to 4 classes.
Found 595 images belonging to 4 classes.
Found 595 images belonging to 4 classes.
Found 595 images belonging to 4 classes.
Found 595 images belong